## Базовый пример запуска Gamac

### Импортируем нужные библиотеки

In [ ]:
import sys
sys.path.append('../')

import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from torchvision.datasets import CIFAR100

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### Получим данные для кластеризации

In [ ]:
# Импортируем датафрейм для табличных данных
data = load_digits(as_frame=True)

table = data['data']
table.head()
targets = data['targets'].values

In [ ]:
# Импортируем данные для CIFAR
# Первая загрузка будет долгая так как он грузит из open-source
cifar100 = CIFAR100('../data/cifar', download=True, train=False)

cifar_txt = [f'a photo of {cifar100.classes[img[1]]}' for img in cifar100][:100]
cifar_img = [img[0] for img in cifar100][:100]
cifar_table = pd.DataFrame(cifar100.targets[:100])

### Инициализируем Gamac
- Класс Gamac имеет следующие аргументы:
    - mab_solver: MabSolvers = MabSolvers.SOFTMAX - тип алгоритма multi-arm bandit
    - hyper_optimiser: HyperOptimisers = HyperOptimisers.OPTUNA - Тип оптимизатора, по умолчанию Optuna
    - target_measures: list[str] | None = None - Целевая мера, по умолчанию не указывается - в этом случае происходит автоматический подбор меры
    - time_limit: int | None = None - Время в секундах поиска оптимальной модели кластеризации
    - iter_limit: int | None = 50 - Кол-во итераций поиска оптимальной модели кластеризации
    - batch_size: int = 32 - размер батча для предобработки текста и изображений
    - model_name: str = "openai/clip-vit-large-patch14" - CLIP-based модель для обработки изображений и текста

In [ ]:
# По умолчанию кол-во итераций стоит 50, в реальных кейсах число лучше ставить больше
gamac = Gamac(iter_limit=30)

### Запустим поиск оптимального алгоритма кластеризации для табличных данных
Работа происходит в три этапа
1. Обработка и приведение модальностей в единый вектор
2. Определение меры - можно задать явно в target_measures, либо оставить пустым (в этом случае мера будет подобрана автоматически)
3. Поиск алгоритма кластеризации с наилучшими по выбранным мерам гиперпараметрами
- В ходе перебора логируются успешные и неуспешные итерации перебора (Success и Failed Run)
- Неуспешные итерации появляются в ходе получения некорректных гиперпараметров алгоритма под конкретный датасет

### Базовый пример для табличных данных
- В данном примере мы используем автоподбор меры

In [ ]:
result = gamac.run(table=table, classes=targets)

### На выходе получаем итоговый датасет после обработки

In [ ]:
print(result.df.shape)
print(result.df[:10])

### Также получаем информацию о лучшей модели - ее наилучшей конфигурацией и финального значения мер

In [ ]:
print(result.estimation) # Значение внутренней меры
print(result.f1_score) # Значение внешней меры при наличии разметки

### Получение меток по лучшему классификатору

In [ ]:
print(result.labels)

### Пример для мультимодальных данных
Для этого используем данные по CIFAR100

### В данном примере можно использовать все модальности (таблица, текст и изображение)
- Также меру можно задать явно в target_measures

In [ ]:
# В данном примере можно использовать все модальности (таблица, текст и изображение)
gamac = Gamac(iter_limit=30, target_measures={Internal.SYM})
result = gamac.run(table=cifar_table, text=cifar_txt, image=cifar_img)

print(result.labels)

### или использовать только текст + изображение

In [ ]:
result = gamac.run(text=cifar_txt, image=cifar_img)

print(result.labels)

### Таким образом Gamac может подбирать кластеризацию для:
1. табличных
2. изображений с текстом
3. совместно